In [2]:
import math
from sklearn.datasets import load_iris
from collections import Counter

# Load Iris dataset
iris = load_iris()

X = iris.data
y = iris.target

features = [
    "Sepal_Length",
    "Sepal_Width",
    "Petal_Length",
    "Petal_Width"
]

classes = iris.target_names


# Entropy
def entropy(y):

    count = Counter(y)
    total = len(y)

    result = 0

    for value in count.values():
        p = value / total
        result -= p * math.log2(p)

    return result


# Information Gain
def information_gain(y, left_y, right_y):

    parent_entropy = entropy(y)

    weighted_entropy = (
        len(left_y) / len(y) * entropy(left_y)
        +
        len(right_y) / len(y) * entropy(right_y)
    )

    return parent_entropy - weighted_entropy


# Find best split
def best_split(X, y):

    best_gain = -1
    best_feature = None
    best_threshold = None

    for feature in range(X.shape[1]):

        values = sorted(set(X[:, feature]))

        for i in range(len(values) - 1):

            threshold = (values[i] + values[i + 1]) / 2

            left = X[:, feature] <= threshold
            right = X[:, feature] > threshold

            left_y = y[left]
            right_y = y[right]

            if len(left_y) == 0 or len(right_y) == 0:
                continue

            gain = information_gain(
                y,
                left_y,
                right_y
            )

            if gain > best_gain:
                best_gain = gain
                best_feature = feature
                best_threshold = threshold

    return best_feature, best_threshold


# ID3
def id3(X, y, depth=0, max_depth=3):

    # Pure node
    if len(set(y)) == 1:
        return classes[y[0]]

    # Maximum depth
    if depth == max_depth:
        majority = Counter(y).most_common(1)[0][0]
        return classes[majority]

    feature, threshold = best_split(X, y)

    if feature is None:
        majority = Counter(y).most_common(1)[0][0]
        return classes[majority]

    left = X[:, feature] <= threshold
    right = X[:, feature] > threshold

    tree = {
        "feature": features[feature],
        "threshold": threshold,
        "left": id3(
            X[left],
            y[left],
            depth + 1,
            max_depth
        ),
        "right": id3(
            X[right],
            y[right],
            depth + 1,
            max_depth
        )
    }

    return tree


# Build tree
tree = id3(X, y)


# Print only tree
def print_tree(tree, space=""):

    if not isinstance(tree, dict):
        print(space + "->", tree)
        return

    print(
        space
        + tree["feature"]
        + " <= "
        + str(round(tree["threshold"], 2))
    )

    print(space + "|-- True")
    print_tree(tree["left"], space + "|   ")

    print(space + "|-- False")
    print_tree(tree["right"], space + "    ")


print_tree(tree)

Petal_Length <= 2.45
|-- True
|   -> setosa
|-- False
    Petal_Width <= 1.75
    |-- True
    |   Petal_Length <= 4.95
    |   |-- True
    |   |   -> versicolor
    |   |-- False
    |       -> virginica
    |-- False
        Petal_Length <= 4.85
        |-- True
        |   -> virginica
        |-- False
            -> virginica
